# Build Analytical Tables

This notebook creates two analytical datasets with different observation levels:

1. **order_sales** — one row per order  
   Used for GMV, order count, AOV, monthly trend and customer-state analysis.

2. **item_sales** — one row per order item  
   Used for product-category and seller contribution analysis.

Payment and item tables are not joined at their raw grains because this would create a many-to-many row explosion.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

orders_clean = spark.table(
    "workspace.olist.clean_orders"
)

order_payments_clean = spark.table(
    "workspace.olist.clean_order_payments"
)

order_items_clean = spark.table(
    "workspace.olist.clean_order_items"
)

products_clean = spark.table(
    "workspace.olist.clean_products"
)

customers_clean = spark.table(
    "workspace.olist.clean_customers"
)

sellers_clean = spark.table(
    "workspace.olist.clean_sellers"
)

## 1. Aggregate payments to order level

The payment table has multiple rows per order because an order may use multiple payment records.

Before joining payments with orders, payments are aggregated to one row per `order_id`.

`booked_gmv` represents the total amount paid by the customer, including freight. It is not Olist's net accounting revenue.

In [0]:
payment_by_order = (
    order_payments_clean
    .groupBy("order_id")
    .agg(
        F.sum(
            F.col("payment_value")
            .cast("decimal(18, 2)")
        ).alias("booked_gmv"),

        F.count("*").alias("payment_record_count"),

        F.countDistinct(
            "payment_type"
        ).alias("payment_type_count"),

        F.max(
            "payment_installments"
        ).alias("max_installments")
    )
)

In [0]:
display(
    payment_by_order.agg(
        F.count("*").alias("rows"),
        F.countDistinct(
            "order_id"
        ).alias("unique_order_ids"),

        F.sum(
            F.col("booked_gmv").isNull().cast("int")
        ).alias("missing_gmv"),

        F.sum(
            (F.col("booked_gmv") <= 0).cast("int")
        ).alias("non_positive_gmv"),

        F.sum(
            "booked_gmv"
        ).alias("total_booked_gmv")
    )
)

rows,unique_order_ids,missing_gmv,non_positive_gmv,total_booked_gmv
97905,97905,0,0,15686469.07


## 2. Build the order-level sales table

Order items are aggregated to one row per order before joining with orders and payments.

This provides order-level item count, merchandise value, freight value and product/seller diversity without changing the order grain.

In [0]:
item_by_order = (
    order_items_clean
    .groupBy("order_id")
    .agg(
        F.count("*").alias("item_count"),

        F.countDistinct(
            "product_id"
        ).alias("product_count"),

        F.countDistinct(
            "seller_id"
        ).alias("seller_count"),

        F.sum(
            F.col("price")
            .cast("decimal(18, 2)")
        ).alias("merchandise_value"),

        F.sum(
            F.col("freight_value")
            .cast("decimal(18, 2)")
        ).alias("freight_value")
    )
    .withColumn(
        "item_gross_value",
        F.col("merchandise_value") +
        F.col("freight_value")
    )
)

In [0]:
display(
    item_by_order.agg(
        F.count("*").alias("rows"),

        F.countDistinct(
            "order_id"
        ).alias("unique_order_ids"),

        F.sum(
            "item_count"
        ).alias("total_item_count"),

        F.sum(
            "merchandise_value"
        ).alias("total_merchandise_value"),

        F.sum(
            "freight_value"
        ).alias("total_freight_value")
    )
)

rows,unique_order_ids,total_item_count,total_merchandise_value,total_freight_value
97905,97905,111752,13449529.68,2234177.06


## 3. Order-Level Sales Table Construction

The order, payment, item, and customer datasets are joined after each source has been reduced to the order grain.

The resulting `order_sales` dataset contains exactly one row per order and supports GMV, order count, AOV, time-trend, and customer-state analysis.

In [0]:
order_sales = (
    orders_clean
    .join(
        payment_by_order,
        on="order_id",
        how="left"
    )
    .join(
        item_by_order,
        on="order_id",
        how="left"
    )
    .join(
        customers_clean.select(
            "customer_id",
            "customer_unique_id",
            "customer_city",
            "customer_state"
        ),
        on="customer_id",
        how="left"
    )
    .withColumn(
        "purchase_date",
        F.to_date(
            "order_purchase_timestamp"
        )
    )
    .withColumn(
        "purchase_month",
        F.to_date(
            F.date_trunc(
                "month",
                F.col("order_purchase_timestamp")
            )
        )
    )
    .select(
        "order_id",
        "customer_id",
        "customer_unique_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "purchase_date",
        "purchase_month",
        "customer_city",
        "customer_state",
        "booked_gmv",
        "payment_record_count",
        "payment_type_count",
        "max_installments",
        "item_count",
        "product_count",
        "seller_count",
        "merchandise_value",
        "freight_value",
        "item_gross_value"
    )
)

Error in callback <bound method UserNamespaceCommandHook.post_run_cell of <dbruntime.DatasetInfo.UserNamespaceCommandHook object at 0xff7db410e660>> (for post_run_cell), with arguments args (<ExecutionResult object at ff7db410ce00, execution_count=9 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at ff7d9c1d1010, raw_cell="order_sales = (
    orders_clean
    .join(
      .." store_history=True silent=False shell_futures=True cell_id=7920121130617415> result=None>,),kwargs {}:


In [0]:
display(
    order_sales.agg(
        F.count("*").alias("rows"),

        F.countDistinct(
            "order_id"
        ).alias("unique_order_ids"),

        F.sum(
            F.col("booked_gmv")
            .isNull()
            .cast("int")
        ).alias("missing_gmv"),

        F.sum(
            F.col("item_count")
            .isNull()
            .cast("int")
        ).alias("missing_items"),

        F.sum(
            F.col("customer_state")
            .isNull()
            .cast("int")
        ).alias("missing_customer_state"),

        F.sum(
            "booked_gmv"
        ).alias("total_booked_gmv"),

        F.sum(
            "item_gross_value"
        ).alias("total_item_gross_value")
    )
)

rows,unique_order_ids,missing_gmv,missing_items,missing_customer_state,total_booked_gmv,total_item_gross_value
97905,97905,0,0,0,15686469.07,15683706.74


## 4. Payment-to-Item Value Reconciliation

`booked_gmv` is compared with merchandise plus freight value at the order level.

A tolerance of 0.01 BRL is used to distinguish material differences from minor rounding differences.

In [0]:
order_value_reconciliation = (
    order_sales
    .withColumn(
        "value_difference",
        F.col("booked_gmv") -
        F.col("item_gross_value")
    )
    .withColumn(
        "absolute_value_difference",
        F.abs(
            F.col("value_difference")
        )
    )
)

In [0]:
value_reconciliation_summary = (
    order_value_reconciliation
    .agg(
        F.count("*").alias("total_orders"),

        F.sum(
            (
                F.col("absolute_value_difference") <= 0.01
            ).cast("int")
        ).alias("orders_within_tolerance"),

        F.sum(
            (
                F.col("absolute_value_difference") > 0.01
            ).cast("int")
        ).alias("orders_with_value_difference"),

        F.sum(
            (
                F.col("value_difference") > 0.01
            ).cast("int")
        ).alias("payment_above_item_total"),

        F.sum(
            (
                F.col("value_difference") < -0.01
            ).cast("int")
        ).alias("payment_below_item_total"),

        F.sum(
            "value_difference"
        ).alias("net_value_difference"),

        F.sum(
            "absolute_value_difference"
        ).alias("total_absolute_difference"),

        F.max(
            "absolute_value_difference"
        ).alias("max_absolute_difference")
    )
)

display(value_reconciliation_summary)

total_orders,orders_within_tolerance,orders_with_value_difference,payment_above_item_total,payment_below_item_total,net_value_difference,total_absolute_difference,max_absolute_difference
97905,97609,296,259,37,2762.33,3033.13,182.81


In [0]:
display(
    order_value_reconciliation
    .filter(
        F.col("absolute_value_difference") > 0.01
    )
    .select(
        "order_id",
        "order_status",
        "booked_gmv",
        "merchandise_value",
        "freight_value",
        "item_gross_value",
        "value_difference",
        "item_count",
        "payment_record_count"
    )
    .orderBy(
        F.desc("absolute_value_difference")
    )
    .limit(20)
)

order_id,order_status,booked_gmv,merchandise_value,freight_value,item_gross_value,value_difference,item_count,payment_record_count
ce6d150fb29ada17d2082f4847107665,delivered,1586.47,1299.00,104.66,1403.66,182.81,1,1
70b742795bc441e94a44a084b6d9ce7a,delivered,578.82,269.99,196.94,466.93,111.89,1,1
996c7e73600ad3723e8627ab7bef81e4,delivered,664.43,559.90,28.00,587.90,76.53,1,1
70b7e94ea46d3e8b5bc12a50186edaf0,delivered,274.84,167.88,45.27,213.15,61.69,3,1
bc2c82b0ef78d2252b6176d1972db7c9,delivered,303.02,165.00,77.01,242.01,61.01,3,1
af9ffff2ce6b3defd34fd4c78857a379,delivered,466.97,395.65,17.52,413.17,53.80,1,1
bfdb5bbb06458d600a33d61f5f287472,delivered,394.36,297.00,51.93,348.93,45.43,1,1
8d9c0dc8d5a2ce804f6b925d8f8e6c1d,delivered,293.89,209.80,44.65,254.45,39.44,2,1
b7579d24f5b2dd3e20f2e57d0e07d170,delivered,504.37,447.00,19.28,466.28,38.09,1,1
abf1130bc676c9dcadf91e24f5e30a30,delivered,496.89,391.60,67.76,459.36,37.53,4,1


### Reconciliation Decision

- 99.70% of orders are within the 0.01 BRL reconciliation tolerance.
- 296 orders have a payment-to-item value difference above the tolerance.
- The net difference represents approximately 0.018% of total booked GMV.
- No orders are excluded.
- `booked_gmv` remains the authoritative monetary KPI.
- For category and seller analysis, booked GMV is allocated proportionally according to each item's price plus freight value.

## 5. Item-Level Sales Table Construction

The `item_sales` dataset contains one row per order item.

Order-level booked GMV is allocated across items according to each item's share of the order's total item gross value. This ensures that category and seller contributions reconcile with total booked GMV.

In [0]:
item_sales = (
    order_items_clean
    .join(
        products_clean.select(
            "product_id",
            "product_category_name"
        ),
        on="product_id",
        how="left"
    )
    .join(
        sellers_clean.select(
            "seller_id",
            "seller_city",
            "seller_state"
        ),
        on="seller_id",
        how="left"
    )
    .join(
        order_sales.select(
            "order_id",
            "order_status",
            "purchase_date",
            "purchase_month",
            "customer_state",
            "booked_gmv",
            F.col("item_gross_value")
            .alias("order_item_gross_value")
        ),
        on="order_id",
        how="left"
    )
    .withColumn(
        "line_gross_value",
        F.col("price") +
        F.col("freight_value")
    )
    .withColumn(
        "allocated_booked_gmv",
        F.when(
            F.col("order_item_gross_value") > 0,
            (
                F.col("booked_gmv").cast("double") *
                F.col("line_gross_value").cast("double")
            ) /
            F.col("order_item_gross_value").cast("double")
        )
    )
    .select(
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "order_status",
        "purchase_date",
        "purchase_month",
        "customer_state",
        "seller_city",
        "seller_state",
        "product_category_name",
        "price",
        "freight_value",
        "line_gross_value",
        "order_item_gross_value",
        "booked_gmv",
        "allocated_booked_gmv"
    )
)

In [0]:
display(
    item_sales.agg(
        F.count("*").alias("rows"),

        F.countDistinct(
            "order_id",
            "order_item_id"
        ).alias("unique_item_keys"),

        F.countDistinct(
            "order_id"
        ).alias("unique_order_ids"),

        F.sum(
            F.col("product_category_name")
            .isNull()
            .cast("int")
        ).alias("missing_category"),

        F.sum(
            F.col("seller_state")
            .isNull()
            .cast("int")
        ).alias("missing_seller_state"),

        F.sum(
            "line_gross_value"
        ).alias("total_line_gross_value"),

        F.sum(
            "allocated_booked_gmv"
        ).alias("total_allocated_booked_gmv")
    )
)

rows,unique_item_keys,unique_order_ids,missing_category,missing_seller_state,total_line_gross_value,total_allocated_booked_gmv
111752,111752,97905,0,0,1.5683706739998762E7,1.5686469069998719E7


## 6. Post-Construction Quality Validation

Allocated booked GMV is aggregated back to the order level and compared with the original order-level booked GMV.

This validation confirms that the item-level allocation preserves the authoritative monetary KPI without introducing join-related duplication or value loss.

In [0]:
allocated_gmv_by_order = (
    item_sales
    .groupBy("order_id")
    .agg(
        F.sum(
            "allocated_booked_gmv"
        ).alias("reconstructed_booked_gmv")
    )
)

In [0]:
allocation_reconciliation = (
    order_sales
    .select(
        "order_id",
        "booked_gmv"
    )
    .join(
        allocated_gmv_by_order,
        on="order_id",
        how="left"
    )
    .withColumn(
        "allocation_difference",
        F.col("reconstructed_booked_gmv") -
        F.col("booked_gmv").cast("double")
    )
    .withColumn(
        "absolute_allocation_difference",
        F.abs(
            F.col("allocation_difference")
        )
    )
)

In [0]:
display(
    allocation_reconciliation.agg(
        F.count("*").alias("total_orders"),

        F.sum(
            F.col("reconstructed_booked_gmv")
            .isNull()
            .cast("int")
        ).alias("orders_without_allocation"),

        F.sum(
            (
                F.col("absolute_allocation_difference") > 0.01
            ).cast("int")
        ).alias("orders_above_tolerance"),

        F.max(
            "absolute_allocation_difference"
        ).alias("max_absolute_difference"),

        F.sum(
            "allocation_difference"
        ).alias("total_allocation_difference")
    )
)

total_orders,orders_without_allocation,orders_above_tolerance,max_absolute_difference,total_allocation_difference
97905,0,0,9.094947017729282E-13,3.983835483722942E-11


## 7. Analytical Table Persistence

The validated analytical datasets are saved as Delta tables for use in the KPI, decomposition, and segment analysis notebooks.

In [0]:
analytical_tables = {
    "order_sales": order_sales,
    "item_sales": item_sales
}

for table_name, dataframe in analytical_tables.items():
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(
            f"workspace.olist.{table_name}"
        )
    )

In [0]:
display(
    spark.sql(
        "SHOW TABLES IN workspace.olist"
    )
    .filter(
        F.col("tableName").isin(
            "order_sales",
            "item_sales"
        )
    )
    .orderBy(
        "tableName"
    )
)

database,tableName,isTemporary
olist,item_sales,false
olist,order_sales,false


## Notebook Outcome

Two validated analytical datasets were created:

### `order_sales`

- Observation level: one row per order
- Rows: 97,905
- Primary uses: booked GMV, order count, AOV, monthly trends, and customer-state analysis

### `item_sales`

- Observation level: one row per order item
- Rows: 111,752
- Primary uses: product-category and seller contribution analysis
- Booked GMV is proportionally allocated across order items

Post-construction validation confirmed:

- No row loss
- No join-induced duplication
- No missing monetary allocation
- No allocation differences above the 0.01 BRL tolerance